---
title: Polars I
---

::: {note} Learning Outcomes
* Build a `DataFrame` from a file, a list of rows, a dictionary of columns, or a `Series`, and describe it with `columns`, `dtypes`, `schema`, and `shape`.
* Extract rows and columns using `[]`, `select`, and `filter`.
* Combine filtering conditions with the bitwise boolean operators.
* Address rows by position, and record those positions with `with_row_index`.
* Summarize a table with utility methods such as `.describe()`, `.sample()`, `.value_counts()`, and `.unique()`.
* Add, modify, rename, and drop columns, and order a table with `.sort()`.
:::

Last time, we met the `Series`: a named, one-dimensional sequence of values that all share a single data type. Almost no dataset arrives as one column, so we now turn to the structure that holds a whole table, the `DataFrame`, and to the operations that get data into and out of it.

We will work with two datasets in this chapter. The first records the results of United States presidential elections; the second records the names given to babies born in California.

In [1]:
# `pl` is the conventional alias for Polars, as `np` is for NumPy
import polars as pl

## `DataFrame`s and `Series`

A `DataFrame` is a two-dimensional table of data with named columns, where each row is identified by its position in the table. Every column of a `DataFrame` is a `Series`, and a `DataFrame` is a collection of `Series` that all have the same length.

A `DataFrame` can be created from scratch or loaded from a file. We'll cover four of the many ways of doing so:

1. From a CSV file.
2. From a list of rows.
3. From a dictionary of columns.
4. From a `Series`.

### From a CSV File

Polars reads a number of file formats. We will use `read_csv` throughout the course to load a comma-separated file into a `DataFrame`.

In [2]:
elections = pl.read_csv("data/elections.csv")
elections

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927
1828,"""John Quincy Adams""","""National Republican""",500897,"""loss""",43.796073
1832,"""Andrew Jackson""","""Democratic""",702735,"""win""",54.574789
…,…,…,…,…,…
2016,"""Jill Stein""","""Green""",1457226,"""loss""",1.073699
2020,"""Joseph Biden""","""Democratic""",81268924,"""win""",51.311515
2020,"""Donald Trump""","""Republican""",74216154,"""loss""",46.858542


The code above stores our `DataFrame` object in the `elections` variable. Upon inspection, our `elections` `DataFrame` has 182 rows and 6 columns (`Year`, `Candidate`, `Party`, `Popular vote`, `Result`, `%`). Each row represents a single record — in our example, a presidential candidate from some particular year. Each column represents a single attribute or feature of the record.

Notice the three lines Polars prints above the data itself. The first gives the shape of the table, the second names the columns, and the third gives the data type of each column: `str` for the text columns, `i64` for whole numbers, `f64` for decimals. Every table you print tells you how big it is and what it holds.

`read_csv` also takes optional arguments that shape the table as it is read.

In [3]:
# `columns` keeps just the columns we name, in the order we name them
pl.read_csv("data/elections.csv", columns=["Candidate", "Year", "%"])

Year,Candidate,%
i64,str,f64
1824,"""Andrew Jackson""",57.210122
1824,"""John Quincy Adams""",42.789878
1828,"""Andrew Jackson""",56.203927
1828,"""John Quincy Adams""",43.796073
1832,"""Andrew Jackson""",54.574789
…,…,…
2016,"""Jill Stein""",1.073699
2020,"""Joseph Biden""",51.311515
2020,"""Donald Trump""",46.858542


In [4]:
# `n_rows` stops reading after the first few rows, which is handy for a very large file
pl.read_csv("data/elections.csv", n_rows=5)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927
1828,"""John Quincy Adams""","""National Republican""",500897,"""loss""",43.796073
1832,"""Andrew Jackson""","""Democratic""",702735,"""win""",54.574789


### From a List of Rows

We'll now explore creating a `DataFrame` with data of our own. The two cells below build the same two-row table of fruit prices, one row at a time.

The first passes a list of lists. `schema` names the columns, and `orient="row"` tells Polars to read each inner list as a row rather than as a column.

In [5]:
df_list_1 = pl.DataFrame(
    [["Kiwi", 5.49],
     ["Orange", 3.99]],
    schema=["Fruit", "Price"], orient="row"
)
df_list_1

Fruit,Price
str,f64
"""Kiwi""",5.49
"""Orange""",3.99


The second passes a list of dictionaries. Each dictionary is a row, and its keys supply the column names, so there is no schema to write out.

In [6]:
df_list_2 = pl.DataFrame(
    [{"Fruit": "Kiwi", "Price": 5.49},
     {"Fruit": "Orange", "Price": 3.99}]
)
df_list_2

Fruit,Price
str,f64
"""Kiwi""",5.49
"""Orange""",3.99


### From a Dictionary of Columns

A dictionary describes the table by column instead of by row: each key is a column name, and each value holds that column's data.

In [7]:
df_dict = pl.DataFrame(
    {"Fruit": ["Kiwi", "Orange"],
     "Price": [5.49, 3.99]}
)
df_dict

Fruit,Price
str,f64
"""Kiwi""",5.49
"""Orange""",3.99


### From a `Series`

Since a `DataFrame` is a collection of equal-length `Series`, we can build one out of `Series` we already have. Consider `ser_a` and `ser_b`.

In [8]:
ser_a = pl.Series("ser_a", ["a1", "a2", "a3"])
ser_b = pl.Series("ser_b", ["b1", "b2", "b3"])
ser_a

ser_a
str
"""a1"""
"""a2"""
"""a3"""


Passing them in a dictionary puts them side by side, under whatever column names we choose.

In [9]:
pl.DataFrame(
    {"ColumnA": ser_a, "ColumnB": ser_b}
)

ColumnA,ColumnB
str,str
"""a1""","""b1"""
"""a2""","""b2"""
"""a3""","""b3"""


A single `Series` makes a one-column `DataFrame`, either by handing it to the constructor or by calling `.to_frame()` on it. Either way, the name of the `Series` becomes the name of the column.

In [10]:
pl.DataFrame(ser_a)

ser_a
str
"""a1"""
"""a2"""
"""a3"""


In [11]:
ser_a.to_frame()

ser_a
str
"""a1"""
"""a2"""
"""a3"""


## `DataFrame` Attributes: `columns`, `dtypes`, and `shape`

Column names in a `DataFrame` are almost always unique. Looking back to the `elections` dataset, it wouldn't make sense to have two columns named `"Candidate"`. Sometimes you'll want to extract the names, the types, or the size of a table rather than the data itself, most often when meeting a dataset for the first time.

For the column names, use `DataFrame.columns`:

In [12]:
elections.columns

['Year', 'Candidate', 'Party', 'Popular vote', 'Result', '%']

For the data type of each column, use `DataFrame.dtypes`. The types come back in the same order as the names above.

In [13]:
elections.dtypes

[Int64, String, String, Int64, String, Float64]

`DataFrame.schema` reports both at once, pairing each column with its type. This is the quickest way to check that a file was read the way you expected: that a column of years arrived as `Int64` rather than as `String`, for instance.

In [14]:
elections.schema

Schema([('Year', Int64),
        ('Candidate', String),
        ('Party', String),
        ('Popular vote', Int64),
        ('Result', String),
        ('%', Float64)])

And for the size of the `DataFrame`, `DataFrame.shape` gives the number of rows followed by the number of columns:

In [15]:
elections.shape

(182, 6)

## Extracting Data from a `DataFrame`

Now that we've learned more about `DataFrame`s, let's dive deeper into their capabilities.

The API (Application Programming Interface) for the `DataFrame` class is enormous. In this section, we'll discuss several methods of the `DataFrame` API that allow us to extract subsets of data.

The simplest way to manipulate a `DataFrame` is to extract a subset of rows and columns, known as **slicing**.

Common ways we may want to extract data are grabbing:

- The first or last `n` rows in the `DataFrame`.
- Data at a certain position.
- Data satisfying some condition.

We will do so with three primary tools of the `DataFrame` class:

1. `.head` and `.tail`
2. `[]`
3. `filter` and `select`

### Extracting Data with `.head` and `.tail`

The simplest scenario in which we want to extract data is when we simply want to select the first or last few rows of the `DataFrame`.

To extract the first `n` rows of a `DataFrame` `df`, we use the syntax `df.head(n)`. Called with no argument at all, `.head` gives us five.

In [16]:
# Extract the first 5 rows of the DataFrame
elections.head()

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927
1828,"""John Quincy Adams""","""National Republican""",500897,"""loss""",43.796073
1832,"""Andrew Jackson""","""Democratic""",702735,"""win""",54.574789


Similarly, calling `df.tail(n)` allows us to extract the last `n` rows of the `DataFrame`.

In [17]:
# Extract the last 5 rows of the DataFrame
elections.tail(5)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
2016,"""Jill Stein""","""Green""",1457226,"""loss""",1.073699
2020,"""Joseph Biden""","""Democratic""",81268924,"""win""",51.311515
2020,"""Donald Trump""","""Republican""",74216154,"""loss""",46.858542
2020,"""Jo Jorgensen""","""Libertarian""",1865724,"""loss""",1.1779795
2020,"""Howard Hawkins""","""Green""",405035,"""loss""",0.255731


### Extraction with `[]`

The `[]` selection operator takes up to two arguments: the first names the rows we want, and the second names the columns. It selects rows by **position**, counting from 0 in the order the rows currently sit in the table, and columns by **label**.

Each argument to `[]` can be:

1. A single value.
2. A list.
3. A slice. A slice of row positions is **exclusive** of its right-hand side, exactly like ordinary Python indexing, while a slice of column labels is **inclusive** of both of its ends.

For example, to select a single value, we can ask for the row at position `0` and the column labeled `Candidate`.

In [18]:
elections[0, "Candidate"]

'Andrew Jackson'

Two single values pick out one cell, and what comes back is the value sitting in it: here, the string `'Andrew Jackson'`.

Two lists pick out a rectangle of the table. The rows arrive in the order we asked for them rather than in table order.

In [19]:
elections[[87, 25, 179], ["Year", "Party", "%"]]

Year,Party,%
i64,str,f64
1932,"""Republican""",39.830594
1860,"""Southern Democratic""",18.138998
2020,"""Republican""",46.858542


A slice of column labels runs from one column to another and includes both ends. `"%"` is the last column of `elections`, and it appears in the result below.

In [20]:
elections[[87, 25, 179], "Popular vote":"%"]

Popular vote,Result,%
i64,str,f64
15761254,"""loss""",39.830594
848019,"""loss""",18.138998
74216154,"""loss""",46.858542


Suppose instead that we want *all* rows and only a few columns. The shorthand `:` is useful for this.

In [21]:
elections[:, ["Year", "Candidate", "Result"]]

Year,Candidate,Result
i64,str,str
1824,"""Andrew Jackson""","""loss"""
1824,"""John Quincy Adams""","""win"""
1828,"""Andrew Jackson""","""win"""
1828,"""John Quincy Adams""","""loss"""
1832,"""Andrew Jackson""","""win"""
…,…,…
2016,"""Jill Stein""","""loss"""
2020,"""Joseph Biden""","""win"""
2020,"""Donald Trump""","""loss"""


We can use the same shorthand to ask for all columns.

In [22]:
elections[[87, 25, 179], :]

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1932,"""Herbert Hoover""","""Republican""",15761254,"""loss""",39.830594
1860,"""John C. Breckinridge""","""Southern Democratic""",848019,"""loss""",18.138998
2020,"""Donald Trump""","""Republican""",74216154,"""loss""",46.858542


A single column label returns that column as a `Series`.

In [23]:
elections[[87, 25, 179], "Popular vote"]

Popular vote
i64
15761254
848019
74216154


Wrapping that same label in a list asks for a table of one column, and a table of one column is what comes back.

In [24]:
elections[[87, 25, 179], ["Popular vote"]]

Popular vote
i64
15761254
848019
74216154


When `[]` is given only one argument, and that argument is a list of integers or a slice, Polars reads it as rows and hands back every column.

In [25]:
elections[[180, 181]]

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
2020,"""Jo Jorgensen""","""Libertarian""",1865724,"""loss""",1.1779795
2020,"""Howard Hawkins""","""Green""",405035,"""loss""",0.255731


A single argument that is a *string*, on the other hand, names a column, and that column comes back as a `Series`. This is the shorthand we use whenever we want one column and nothing else, and it shows up throughout the rest of the chapter.

In [26]:
elections["Candidate"]

Candidate
str
"""Andrew Jackson"""
"""John Quincy Adams"""
"""Andrew Jackson"""
"""John Quincy Adams"""
"""Andrew Jackson"""
…
"""Jill Stein"""
"""Joseph Biden"""
"""Donald Trump"""


#### Selecting Columns by Position

The second argument to `[]` accepts **column numbers** as well as column labels. The numbers count from the left edge of the table, starting at 0, so `elections[:, 1]` and `elections[:, "Candidate"]` name the same column.

Slicing by column number, like slicing by row position, is **exclusive** of the right-hand side of the slice. The inclusive behavior we saw above belongs to label slices only.

In [27]:
# Extracting the value at the first row (row 0) and the second column
# Remember that Python indexing begins at position 0!
elections[0, 1]

'Andrew Jackson'

In [28]:
# Extracting the second, third, and fourth rows of the second column
# (returns a Series, since we asked for a single column)
elections[[1, 2, 3], 1]

Candidate
str
"""John Quincy Adams"""
"""Andrew Jackson"""
"""John Quincy Adams"""


In [29]:
# Select the rows at positions 1, 2, and 3
# Select the columns at positions 0, 1, and 2
elections[[1, 2, 3], [0, 1, 2]]

Year,Candidate,Party
i64,str,str
1824,"""John Quincy Adams""","""Democratic-Republican"""
1828,"""Andrew Jackson""","""Democratic"""
1828,"""John Quincy Adams""","""National Republican"""


In [30]:
# A list of row positions and a slice of column numbers
# The column at position 3 is left out, since number slices are exclusive
elections[[1, 2, 3], 0:3]

Year,Candidate,Party
i64,str,str
1824,"""John Quincy Adams""","""Democratic-Republican"""
1828,"""Andrew Jackson""","""Democratic"""
1828,"""John Quincy Adams""","""National Republican"""


In [31]:
# One argument, so Polars reads it as rows and returns all columns
elections[138:144]

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1988,"""Ron Paul""","""Libertarian""",431750,"""loss""",0.47266
1992,"""Andre Marrou""","""Libertarian""",290087,"""loss""",0.278516
1992,"""Bill Clinton""","""Democratic""",44909806,"""win""",43.118485
1992,"""Bo Gritz""","""Populist""",106152,"""loss""",0.101918
1992,"""George H. W. Bush""","""Republican""",39104550,"""loss""",37.544784
1992,"""Ross Perot""","""Independent""",19743821,"""loss""",18.956298


A whole row on its own comes back from `.row()`, as a tuple of values in column order.

In [32]:
elections.row(0)

(1824, 'Andrew Jackson', 'Democratic-Republican', 151271, 'loss', 57.21012204)

Passing `named=True` gives a dictionary instead, which is much easier to read when a table is wide.

In [33]:
elections.row(0, named=True)

{'Year': 1824,
 'Candidate': 'Andrew Jackson',
 'Party': 'Democratic-Republican',
 'Popular vote': 151271,
 'Result': 'loss',
 '%': 57.21012204}

A row's position is the only handle we have on it, and that position belongs to the table rather than to the row. Any operation that reorders the table therefore hands out new positions, which is a point we return to at the end of the section.

### Extraction with `filter` and `select`

`[]` asks for positions and labels, which makes it concise for the quick looks at a table we take constantly. Anything *computed*, though (a condition, an arithmetic result) goes through a second pair of methods.

`filter` chooses rows and `select` chooses columns. Both take **expressions**, which are built with `pl.col` and can compare and combine columns before anything is returned. This is the pairing you'll reach for most often, because a condition like "more than 60 million popular votes" describes the rows you want without needing to know where they sit.

Two rules cover most of the confusion: `filter` narrows the rows and leaves every column in place, and `select` decides which columns come back.

In [34]:
# select takes a list of column names and returns a DataFrame
elections.select(["Year", "Candidate", "Result"])

Year,Candidate,Result
i64,str,str
1824,"""Andrew Jackson""","""loss"""
1824,"""John Quincy Adams""","""win"""
1828,"""Andrew Jackson""","""win"""
1828,"""John Quincy Adams""","""loss"""
1832,"""Andrew Jackson""","""win"""
…,…,…
2016,"""Jill Stein""","""loss"""
2020,"""Joseph Biden""","""win"""
2020,"""Donald Trump""","""loss"""


`select` also accepts a computed expression. `pl.col("Popular vote")` refers to that column, arithmetic on it applies to every value, and `.alias` names the result.

In [35]:
elections.select((pl.col("Popular vote") / 1_000_000).alias("Popular vote (millions)"))

Popular vote (millions)
f64
0.151271
0.113142
0.642806
0.500897
0.702735
…
1.457226
81.268924
74.216154


`filter` takes a condition and returns the rows that satisfy it. Eight candidacies in the dataset drew more than 60 million popular votes, the earliest of them in 2004.

In [36]:
elections.filter(pl.col("Popular vote") > 60000000)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
2004,"""George W. Bush""","""Republican""",62040610,"""win""",50.771824
2008,"""Barack Obama""","""Democratic""",69498516,"""win""",53.02351
2012,"""Barack Obama""","""Democratic""",65915795,"""win""",51.258484
2012,"""Mitt Romney""","""Republican""",60933504,"""loss""",47.384076
2016,"""Donald Trump""","""Republican""",62984828,"""win""",46.407862
2016,"""Hillary Clinton""","""Democratic""",65853514,"""loss""",48.521539
2020,"""Joseph Biden""","""Democratic""",81268924,"""win""",51.311515
2020,"""Donald Trump""","""Republican""",74216154,"""loss""",46.858542


Each of these methods returns a `DataFrame`, so the two can be chained: filter the rows first, then pick the columns to keep.

In [37]:
elections.filter(pl.col("Year") == 2008).select(["Year", "Candidate"])

Year,Candidate
i64,str
2008,"""Barack Obama"""
2008,"""Bob Barr"""
2008,"""Chuck Baldwin"""
2008,"""Cynthia McKinney"""
2008,"""John McCain"""
2008,"""Ralph Nader"""


### Boolean Operators

To filter on more than one condition at a time, we combine boolean masks using **bitwise operators**. In the table below, p and q are boolean expressions.

Symbol | Usage      | Meaning
------ | ---------- | -------------------------------------
~    | ~p       | Returns negation of p
&#124; | p &#124; q | p OR q
&    | p & q    | p AND q
^  | p ^ q | p XOR q (exclusive or)

**Always** wrap each individual condition in a set of parentheses `()` when combining them. Python binds `&` and `|` more tightly than the comparison operators, so `pl.col("Year") == 2008 | pl.col("%") >= 60` is read as `pl.col("Year") == (2008 | pl.col("%")) >= 60` and raises a `TypeError` instead of filtering anything.

For example, to return every candidacy from 2008 *or* with at least 60% of the popular vote, we can write:

In [38]:
# Grab rows from 2008 OR candidates winning over 60% of the vote (or both)
elections.filter((pl.col("Year") == 2008) | (pl.col("%") >= 60))

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1920,"""Warren Harding""","""Republican""",16144093,"""win""",60.574501
1936,"""Franklin Roosevelt""","""Democratic""",27752648,"""win""",60.978107
1964,"""Lyndon Johnson""","""Democratic""",43127041,"""win""",61.344703
1972,"""Richard Nixon""","""Republican""",47168710,"""win""",60.907806
2008,"""Barack Obama""","""Democratic""",69498516,"""win""",53.02351
2008,"""Bob Barr""","""Libertarian""",523715,"""loss""",0.399565
2008,"""Chuck Baldwin""","""Constitution""",199750,"""loss""",0.152398
2008,"""Cynthia McKinney""","""Green""",161797,"""loss""",0.123442
2008,"""John McCain""","""Republican""",59948323,"""loss""",45.737243


Ten rows satisfy that condition: the six candidates who stood in 2008, plus the four landslide winners of 1920, 1936, 1964, and 1972.

If we want the rows where *both* conditions hold, we use `&`.

In [39]:
# Grab post-2000 winners: rows where the year is after 2000 AND the result is a win
elections.filter((pl.col("Year") > 2000) & (pl.col("Result") == "win"))

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
2004,"""George W. Bush""","""Republican""",62040610,"""win""",50.771824
2008,"""Barack Obama""","""Democratic""",69498516,"""win""",53.02351
2012,"""Barack Obama""","""Democratic""",65915795,"""win""",51.258484
2016,"""Donald Trump""","""Republican""",62984828,"""win""",46.407862
2020,"""Joseph Biden""","""Democratic""",81268924,"""win""",51.311515


Note that we need the bitwise operators here, not Python's `and` and `or`. Those two ask a single
yes-or-no question about the whole object, and an expression stands for a column of many values,
so there is no one answer to give:

In [40]:
# This line of code will raise a TypeError
elections.filter((pl.col("Year") == 2008) and (pl.col("%") >= 60))

TypeError: the truth value of an Expr is ambiguous

You probably got here by using a Python standard library function instead of the native expressions API.
Here are some things you might want to try:
- instead of `pl.col('a') and pl.col('b')`, use `pl.col('a') & pl.col('b')`
- instead of `pl.col('a') in [y, z]`, use `pl.col('a').is_in([y, z])`
- instead of `max(pl.col('a'), pl.col('b'))`, use `pl.max_horizontal(pl.col('a'), pl.col('b'))`


Conditions can be strung together as far as we need. Wrapping the whole call in parentheses lets us break a long one across several lines, which is worth doing well before it becomes hard to read.

In [41]:
# To make code more readable, use multiple lines
elections.filter(
    (pl.col("Year") < 2000) &
    (pl.col("Year") > 1941) &
    (pl.col("Result") == "win") &
    (pl.col("%") >= 55)
)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1952,"""Dwight Eisenhower""","""Republican""",34075529,"""win""",55.325173
1956,"""Dwight Eisenhower""","""Republican""",35579180,"""win""",57.650654
1964,"""Lyndon Johnson""","""Democratic""",43127041,"""win""",61.344703
1972,"""Richard Nixon""","""Republican""",47168710,"""win""",60.907806
1984,"""Ronald Reagan""","""Republican""",54455472,"""win""",59.023326


### Working with Row Positions

A row is identified by its position in the table, counting from 0. Those positions are not stored anywhere; `with_row_index` writes them into a column of their own when we want to keep them.

This matters as soon as we reorder a table. Below, we record each row's position and *then* sort by vote share, so the new first column says where each row started out. Lyndon Johnson's 1964 landslide is the largest share in the dataset, and it came from position 114.

In [42]:
# with_row_index adds a column holding each row's current position
elections.with_row_index("original_position").sort("%", descending=True).head()

original_position,Year,Candidate,Party,Popular vote,Result,%
u32,i64,str,str,i64,str,f64
114,1964,"""Lyndon Johnson""","""Democratic""",43127041,"""win""",61.344703
91,1936,"""Franklin Roosevelt""","""Democratic""",27752648,"""win""",60.978107
120,1972,"""Richard Nixon""","""Republican""",47168710,"""win""",60.907806
79,1920,"""Warren Harding""","""Republican""",16144093,"""win""",60.574501
133,1984,"""Ronald Reagan""","""Republican""",54455472,"""win""",59.023326


`with_row_index` returns a new table rather than changing the one we called it on, so `elections` itself still has its original six columns and its original order.

In [43]:
elections.head(3)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927


Adding the index column *after* the sort numbers the rows in their new order instead, counting 0, 1, 2 down the sorted table. Which of the two you want depends on whether you care where a row came from or where it now sits.

In [44]:
elections.sort("%", descending=True).with_row_index().head()

index,Year,Candidate,Party,Popular vote,Result,%
u32,i64,str,str,i64,str,f64
0,1964,"""Lyndon Johnson""","""Democratic""",43127041,"""win""",61.344703
1,1936,"""Franklin Roosevelt""","""Democratic""",27752648,"""win""",60.978107
2,1972,"""Richard Nixon""","""Republican""",47168710,"""win""",60.907806
3,1920,"""Warren Harding""","""Republican""",16144093,"""win""",60.574501
4,1984,"""Ronald Reagan""","""Republican""",54455472,"""win""",59.023326


## The `babynames` Dataset

The rest of this chapter works with a second dataset: the names given to babies born in California, as recorded by the Social Security Administration. Each row holds one name, in one year, for one sex, along with the number of babies who were given it.

The cell below downloads the data and loads it into a `DataFrame`. The code is outside the scope of Data 100, but you're encouraged to dig into it if you are interested.

````{dropdown} Click to see the code
:open: false
```python
# This code pulls census data and loads it into a DataFrame
# We won't cover it explicitly in this class, but you are welcome to explore it on your own
import urllib.request
import os.path
import zipfile

data_url = "https://www.ssa.gov/oact/babynames/state/namesbystate.zip"
local_filename = "data/babynamesbystate.zip"
if not os.path.exists(local_filename): # If the data exists don't download again
    with urllib.request.urlopen(data_url) as resp, open(local_filename, 'wb') as f:
        f.write(resp.read())

zf = zipfile.ZipFile(local_filename, 'r')

ca_name = 'STATE.CA.TXT'
field_names = ['State', 'Sex', 'Year', 'Name', 'Count']
with zf.open(ca_name) as fh:
    babynames = pl.read_csv(fh, has_header=False, new_columns=field_names)

babynames.head()
```
````

In [45]:
# This code pulls census data and loads it into a DataFrame
# We won't cover it explicitly in this class, but you are welcome to explore it on your own
import urllib.request
import os.path
import zipfile

data_url = "https://www.ssa.gov/oact/babynames/state/namesbystate.zip"
local_filename = "data/babynamesbystate.zip"
if not os.path.exists(local_filename): # If the data exists don't download again
    with urllib.request.urlopen(data_url) as resp, open(local_filename, 'wb') as f:
        f.write(resp.read())

zf = zipfile.ZipFile(local_filename, 'r')

ca_name = 'STATE.CA.TXT'
field_names = ['State', 'Sex', 'Year', 'Name', 'Count']
with zf.open(ca_name) as fh:
    babynames = pl.read_csv(fh, has_header=False, new_columns=field_names)

babynames.head()

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1910,"""Mary""",295
"""CA""","""F""",1910,"""Helen""",239
"""CA""","""F""",1910,"""Dorothy""",220
"""CA""","""F""",1910,"""Margaret""",163
"""CA""","""F""",1910,"""Frances""",134


## More Ways to Build a Filter

A boolean expression can describe any condition we can write down, but a long list of alternatives gets verbose in a hurry. Suppose we want every row whose name is one of four we care about.

In [46]:
# Note: The parentheses surrounding the code make it possible to
# break the code into multiple lines for readability. But this is
# still a lot of code just to check for four names...
(
    babynames.filter((pl.col("Name") == "Bella") |
                     (pl.col("Name") == "Alex") |
                     (pl.col("Name") == "Narges") |
                     (pl.col("Name") == "Lisa"))
)

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1923,"""Bella""",5
"""CA""","""F""",1925,"""Bella""",8
"""CA""","""F""",1932,"""Lisa""",5
"""CA""","""F""",1936,"""Lisa""",8
"""CA""","""F""",1939,"""Lisa""",5
…,…,…,…,…
"""CA""","""M""",2018,"""Alex""",495
"""CA""","""M""",2019,"""Alex""",438
"""CA""","""M""",2020,"""Alex""",379


Fortunately, Polars offers more concise ways of saying the same thing.

The `.is_in()` method checks each value of a column against a sequence of values (a list, an array, or another `Series`). It returns the same 317 rows as the four-way condition above, in one line.

In [47]:
names = ["Bella", "Alex", "Narges", "Lisa"]
babynames.filter(pl.col("Name").is_in(names))

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1923,"""Bella""",5
"""CA""","""F""",1925,"""Bella""",8
"""CA""","""F""",1932,"""Lisa""",5
"""CA""","""F""",1936,"""Lisa""",8
"""CA""","""F""",1939,"""Lisa""",5
…,…,…,…,…
"""CA""","""M""",2018,"""Alex""",495
"""CA""","""M""",2019,"""Alex""",438
"""CA""","""M""",2020,"""Alex""",379


String columns carry a whole family of methods under `.str`. `.str.starts_with()` checks the beginning of each string, so the filter below keeps every row whose name begins with the letter `N`.

In [48]:
# Extracting names that begin with the letter "N"
babynames.filter(pl.col("Name").str.starts_with("N"))

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1910,"""Norma""",23
"""CA""","""F""",1910,"""Nellie""",20
"""CA""","""F""",1910,"""Nina""",11
"""CA""","""F""",1910,"""Nora""",6
"""CA""","""F""",1911,"""Nellie""",23
…,…,…,…,…
"""CA""","""M""",2022,"""Nilan""",5
"""CA""","""M""",2022,"""Niles""",5
"""CA""","""M""",2022,"""Nolen""",5


## Useful Utility Functions

Polars contains an extensive library of functions that can help shorten the process of setting and getting information from its data structures. In the following section, we will give overviews of each of the main utility functions that will help us in Data 100.

Discussing all of the functionality offered by Polars could take an entire semester! We will walk you through the most commonly used functions and encourage you to explore and experiment on your own.

- Aggregation methods
- `.shape`, `.height`, and `.width`
- `.describe()`
- `.sample()`
- `.value_counts()`
- `.unique()`

The Polars [documentation](https://docs.pola.rs/api/python/stable/reference/index.html) will be a valuable resource in Data 100 and beyond.

### Aggregation Methods

The array functions you encountered in [Data 8](https://www.data8.org/su23/reference/#array-functions-and-methods) live here as methods you call on a `Series` itself. Below, we pull out the number of babies named Yash in each year the name was recorded.

In [49]:
# Pull out the number of babies named Yash each year
yash_counts = babynames.filter(pl.col("Name") == "Yash")["Count"]
yash_counts

Count
i64
8
9
11
12
10
…
10
9
15


The name appears in 28 rows of the table, and `.mean()` averages the counts across them.

In [50]:
# Average number of babies named Yash each year
# Keep in mind that even if Python gives you 10 decimal places of precision,
# you should think carefully about how much precision is meaningful!
# In this case, one decimal place or even no decimal places would be appropriate.
yash_counts.mean()

17.142857142857142

In [51]:
# Max number of babies named Yash born in any single year
yash_counts.max()

29

### `.shape`, `.height`, and `.width`

These attributes measure the "amount" of data stored in a `DataFrame`. Calling `.shape` returns a tuple containing the number of rows followed by the number of columns.

Many functions strictly require the dimensions of their arguments to match. Asking the table for its dimensions is much faster than counting the items by hand.

In [52]:
# Return the shape of the DataFrame, in the format (num_rows, num_columns)
babynames.shape

(407428, 5)

`.height` and `.width` report those same two numbers one at a time, so multiplying them gives the total number of values the table holds.

In [53]:
# The total number of entries in the object, equal to num_rows * num_columns
babynames.height * babynames.width

2037140

Calling `len` on a `DataFrame` gives its height, which is the number we want far more often than the other two.

In [54]:
# Return the number of rows in the DataFrame
len(babynames)

407428

### `.describe()`

If many statistics are required from a `DataFrame` (minimum value, maximum value, mean value, etc.), then `.describe()` [(documentation)](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.describe.html) can be used to compute all of them at once.

In [55]:
babynames.describe()

statistic,State,Sex,Year,Name,Count
str,str,str,f64,str,f64
"""count""","""407428""","""407428""",407428.0,"""407428""",407428.0
"""null_count""","""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,1985.733609,null,79.543456
"""std""",null,null,27.00766,null,293.698654
"""min""","""CA""","""F""",1910.0,"""Aadan""",5.0
"""25%""",null,null,1969.0,null,7.0
"""50%""",null,null,1992.0,null,13.0
"""75%""",null,null,2008.0,null,38.0
"""max""","""CA""","""M""",2022.0,"""Zyrus""",8260.0


The statistics come back as rows, labeled by the `statistic` column on the left, with one column of results per column of the original table. Text columns are described too: they report a count, a null count, and their alphabetical minimum and maximum, and carry `null` wherever a statistic makes no sense for them.

A few things stand out. No value anywhere in the table is missing, since `null_count` is 0 across the board. The years run from 1910 to 2022. And the smallest `Count` in the dataset is 5, so names rarer than that never made it into the file.

A `Series` can describe itself in the same way, reporting the statistics that suit its data type.

In [56]:
babynames["Sex"].describe()

statistic,value
str,str
"""count""","""407428"""
"""null_count""","""0"""
"""min""","""F"""
"""max""","""M"""


### `.sample()`

As we will see later in the semester, random processes are at the heart of many data science techniques (for example, train-test splits, bootstrapping, and cross-validation). `.sample()` [(documentation)](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.sample.html) lets us quickly select random rows of a `DataFrame`.

By default, `.sample()` selects rows *without* replacement. Pass in the argument `with_replacement=True` to sample with replacement.

In [57]:
# Randomly sample a row from the DataFrame
babynames.sample()

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""M""",1960,"""Eric""",1471


Naturally, this can be chained with the extraction tools from earlier in the chapter.

In [58]:
# Sample 5 random rows, and keep all columns from position 2 onwards
babynames.sample(5)[:, 2:]

Year,Name,Count
i64,str,i64
1930,"""Barbara""",5
1976,"""Shilo""",11
2022,"""Lincoln""",270
1992,"""Abelardo""",27
1970,"""Darron""",14


Wrapping a chain of methods in parentheses lets us spread it across several lines. Here we narrow the table to the year 2000, sample four of those rows with replacement, and keep the last three columns.

In [59]:
result = (
    babynames.filter(pl.col("Year") == 2000)
    .sample(4, with_replacement=True)[:, 2:]
)
result

Year,Name,Count
i64,str,i64
2000,"""Sanika""",7
2000,"""Gabriele""",6
2000,"""Edie""",8
2000,"""Keilan""",5


::: {tip}
Rerun any of the cells above and you'll get different rows each time. Pass `seed=` to `.sample()` when you need the same rows on every run, which is most of the time once other people have to reproduce your results.
:::

### `.value_counts()`

The `Series.value_counts()` [(documentation)](https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.value_counts.html) method counts the number of occurrences of each unique value in a `Series`. In other words, it *counts* the number of times each unique *value* appears. This is often useful for determining the most or least common entries in a `Series`.

In [60]:
babynames["Sex"].value_counts()

Sex,count
str,u32
"""F""",239537
"""M""",167891


The result is a two-column `DataFrame`: the distinct values, in a column that keeps the name of the original `Series`, and their counts, in a column named `count`. Those rows come back in no particular order, so pass `sort=True` when the ranking is what you are after.

Below, we count the number of times each name appears in the `"Name"` column, which tells us the name recorded in the most sex-and-year combinations.

In [61]:
babynames["Name"].value_counts(sort=True).head()

Name,count
str,u32
"""Jean""",223
"""Francis""",221
"""Guadalupe""",218
"""Jessie""",217
"""Marion""",214


`Jean` leads with 223 rows: 223 separate combinations of a sex and a year in which at least five California babies were given that name.

### `.unique()`

If we have a `Series` with many repeated values, then `.unique()` [(documentation)](https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.unique.html) can be used to identify only the *unique* values. Here we return every name in `babynames`.

In [62]:
babynames["Name"].unique()

Name
str
"""Zailyn"""
"""Cyrille"""
"""Shivansh"""
"""Alexa"""
"""Allexa"""
…
"""Jacinto"""
"""Daryan"""
"""Aidenn"""


The 407,428 rows of the table hold 20,437 distinct names between them, a count that `.n_unique()` reports directly.

In [63]:
babynames["Name"].n_unique()

20437

The unique values arrive in no particular order. When the order matters, `maintain_order=True` returns them in the order they first appear in the `Series`, which here means starting from the top of the table.

In [64]:
babynames["Name"].unique(maintain_order=True).head(5)

Name
str
"""Mary"""
"""Helen"""
"""Dorothy"""
"""Margaret"""
"""Frances"""


## Adding, Removing, and Modifying Columns

In many data science tasks, we may need to change the columns contained in our `DataFrame` in some way. Fortunately, the syntax to do so is fairly straightforward.

To add a new column, hand `.with_columns()` [(documentation)](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.with_columns.html) a `Series` or an expression under the name we want it to have. Writing that name as a keyword argument, as below, is the most direct way to say it.

In [65]:
# Create a Series of the length of each name
babyname_lengths = babynames["Name"].str.len_chars()

# Add a column named "name_lengths" that includes the length of each name
babynames = babynames.with_columns(name_lengths=babyname_lengths)
babynames.head()

State,Sex,Year,Name,Count,name_lengths
str,str,i64,str,i64,u32
"""CA""","""F""",1910,"""Mary""",295,4
"""CA""","""F""",1910,"""Helen""",239,5
"""CA""","""F""",1910,"""Dorothy""",220,7
"""CA""","""F""",1910,"""Margaret""",163,8
"""CA""","""F""",1910,"""Frances""",134,7


If we need to later modify an existing column, we pass the new values to `.with_columns()` under that column's existing name. Inside the expression, `pl.col("name_lengths")` refers to the column as it stands right now.

In [66]:
# Modify the "name_lengths" column to be one less than its original value
babynames = babynames.with_columns(name_lengths=pl.col("name_lengths") - 1)
babynames.head()

State,Sex,Year,Name,Count,name_lengths
str,str,i64,str,i64,u32
"""CA""","""F""",1910,"""Mary""",295,3
"""CA""","""F""",1910,"""Helen""",239,4
"""CA""","""F""",1910,"""Dorothy""",220,6
"""CA""","""F""",1910,"""Margaret""",163,7
"""CA""","""F""",1910,"""Frances""",134,6


We can rename a column using the `.rename()` method. It takes in a dictionary that maps old column names to their new ones.

In [67]:
# Rename "name_lengths" to "Length"
babynames = babynames.rename({"name_lengths": "Length"})
babynames.head()

State,Sex,Year,Name,Count,Length
str,str,i64,str,i64,u32
"""CA""","""F""",1910,"""Mary""",295,3
"""CA""","""F""",1910,"""Helen""",239,4
"""CA""","""F""",1910,"""Dorothy""",220,6
"""CA""","""F""",1910,"""Margaret""",163,7
"""CA""","""F""",1910,"""Frances""",134,6


If we want to remove a column of a `DataFrame`, we can call the `.drop()` [(documentation)](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.drop.html) method with the name of the column to remove. Dropping rows, by contrast, is a job for `filter`.

In [68]:
# Drop our new "Length" column from the DataFrame
babynames = babynames.drop("Length")
babynames.head()

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1910,"""Mary""",295
"""CA""","""F""",1910,"""Helen""",239
"""CA""","""F""",1910,"""Dorothy""",220
"""CA""","""F""",1910,"""Margaret""",163
"""CA""","""F""",1910,"""Frances""",134


Notice that each of the cells above *re-assigned* `babynames` to the result of the call. This is a subtle but important point: table operations **do not occur in place**. `.with_columns()`, `.rename()`, and `.drop()` each build a new table and hand it back, leaving the table they were called on exactly as it was. The same is true of `filter`, `select`, `.sort()`, and `with_row_index`, which is why `elections` was unchanged earlier in the chapter.

In other words, if we simply call:

In [69]:
# This produces a new table without the column "Name"...
babynames.drop("Name")

# ...but the original `babynames` is unchanged!
# Notice that the "Name" column is still present
babynames.head()

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1910,"""Mary""",295
"""CA""","""F""",1910,"""Helen""",239
"""CA""","""F""",1910,"""Dorothy""",220
"""CA""","""F""",1910,"""Margaret""",163
"""CA""","""F""",1910,"""Frances""",134


## Sorting

Ordering a `DataFrame` can be useful for isolating extreme values. For example, the first 5 rows of a table sorted in descending order (that is, from highest to lowest) hold the 5 largest values. `.sort()` [(documentation)](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.sort.html) orders a `DataFrame` by a column we name. It sorts from lowest to highest unless we ask otherwise with `descending=True`.

In [70]:
# Sort the "Count" column from lowest to highest
babynames.sort("Count").head()

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1910,"""Adelaide""",5
"""CA""","""F""",1910,"""Adele""",5
"""CA""","""F""",1910,"""Adrienne""",5
"""CA""","""F""",1910,"""Althea""",5
"""CA""","""F""",1910,"""Antonia""",5


In [71]:
# Sort the "Count" column from highest to lowest
babynames.sort("Count", descending=True).head()

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""M""",1957,"""Michael""",8260
"""CA""","""M""",1956,"""Michael""",8258
"""CA""","""M""",1990,"""Michael""",8246
"""CA""","""M""",1969,"""Michael""",8245
"""CA""","""M""",1970,"""Michael""",8196


There are a lot of Michaels in California: all five of the largest counts in the dataset belong to that name, topping out at 8,260 babies in 1957.

A `Series` sorts the same way. There is no column to name, since a `Series` is a single column, and only its values come back in their new order.

In [72]:
# Sort the "Name" Series alphabetically
babynames["Name"].sort().head(5)

Name
str
"""Aadan"""
"""Aadan"""
"""Aadan"""
"""Aadarsh"""
"""Aaden"""


::: {warning}
`.sort()` places null values **first**, ahead of every real value, in both sort directions. A `.head()` or a positional slice taken straight after a sort will therefore pick up missing values and push out the rows you were after. Nothing about that is an error, so nothing announces it. Pass `nulls_last=True` whenever a sort feeds a `.head()`, a `.tail()`, or a slice, unless you already know the column holds no nulls — as is the case for both datasets in this chapter.
:::

`babynames` has no missing values, so the small table below has one instead.

In [73]:
demo = pl.DataFrame({"Name": ["Aaliyah", "Bao", "Cyrus"], "Count": [3, None, 1]})
demo.sort("Count", descending=True)

Name,Count
str,i64
"""Bao""",null
"""Aaliyah""",3
"""Cyrus""",1


Sorting from highest to lowest put the missing count at the top. `nulls_last=True` sends it to the bottom, where it stays out of the way of a `.head()`.

In [74]:
demo.sort("Count", descending=True, nulls_last=True)

Name,Count
str,i64
"""Aaliyah""",3
"""Cyrus""",1
"""Bao""",null


## Custom Sorts

Now, let's try to solve a sorting problem using different approaches. Assume we want to find the longest baby names and sort our data accordingly.

### Approach 1: Create a Temporary Column

One method to do this is to first start by creating a column that contains the lengths of the names.

In [75]:
# Create a Series of the length of each name
babyname_lengths = babynames["Name"].str.len_chars()

# Add a column named "name_lengths" that includes the length of each name
babynames = babynames.with_columns(name_lengths=babyname_lengths)
babynames.head(5)

State,Sex,Year,Name,Count,name_lengths
str,str,i64,str,i64,u32
"""CA""","""F""",1910,"""Mary""",295,4
"""CA""","""F""",1910,"""Helen""",239,5
"""CA""","""F""",1910,"""Dorothy""",220,7
"""CA""","""F""",1910,"""Margaret""",163,8
"""CA""","""F""",1910,"""Frances""",134,7


We can then sort the `DataFrame` by that column using `.sort()`:

In [76]:
# Sort by the temporary column
babynames = babynames.sort(by="name_lengths", descending=True)
babynames.head(5)

State,Sex,Year,Name,Count,name_lengths
str,str,i64,str,i64,u32
"""CA""","""F""",1986,"""Mariadelosangel""",5,15
"""CA""","""M""",1987,"""Franciscojavier""",5,15
"""CA""","""M""",1988,"""Franciscojavier""",10,15
"""CA""","""M""",1989,"""Franciscojavier""",6,15
"""CA""","""M""",1991,"""Ryanchristopher""",7,15


The longest names in the dataset run to 15 characters. Finally, we can drop the `name_lengths` column from `babynames` to prevent our table from getting cluttered.

In [77]:
# Drop the "name_lengths" column
babynames = babynames.drop("name_lengths")
babynames.head(5)

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1986,"""Mariadelosangel""",5
"""CA""","""M""",1987,"""Franciscojavier""",5
"""CA""","""M""",1988,"""Franciscojavier""",10
"""CA""","""M""",1989,"""Franciscojavier""",6
"""CA""","""M""",1991,"""Ryanchristopher""",7


### Approach 2: Sorting on an Expression

Another way to approach this is to hand `.sort()` an expression instead of a column name. The sort key is then computed on the way in, so there is no temporary column to create and no temporary column to drop.

In [78]:
babynames.sort(pl.col("Name").str.len_chars(), descending=True).head()

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1986,"""Mariadelosangel""",5
"""CA""","""M""",1987,"""Franciscojavier""",5
"""CA""","""M""",1988,"""Franciscojavier""",10
"""CA""","""M""",1989,"""Franciscojavier""",6
"""CA""","""M""",1991,"""Ryanchristopher""",7


### Approach 3: Sorting with `map_elements`

We can also use `map_elements` [(documentation)](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.map_elements.html) if we want to sort by an arbitrarily defined Python function. Say we want to sort the `babynames` table by the number of `"dr"`s and `"ea"`s in each `"Name"`. We'll define the function `dr_ea_count` to help us out.

`map_elements` hands each value of the column to that function, one value at a time, and collects the results. `return_dtype` tells Polars what type those results will have.

In [79]:
# First, define a function to count the number of times
# "dr" or "ea" appear in each name
def dr_ea_count(string):
    return string.count('dr') + string.count('ea')

# Then, use map_elements to apply dr_ea_count to each name in the "Name" column
babynames = babynames.with_columns(
    dr_ea_count=pl.col("Name").map_elements(dr_ea_count, return_dtype=pl.Int64)
)

# Sort the DataFrame by the new "dr_ea_count" column so we can see our handiwork
babynames = babynames.sort(by="dr_ea_count", descending=True)
babynames.head()

State,Sex,Year,Name,Count,dr_ea_count
str,str,i64,str,i64,i64
"""CA""","""F""",1986,"""Deandrea""",6,3
"""CA""","""F""",1988,"""Deandrea""",5,3
"""CA""","""F""",1990,"""Deandrea""",5,3
"""CA""","""F""",1994,"""Leandrea""",5,3
"""CA""","""M""",1985,"""Deandrea""",6,3


Because it runs Python code once per row, `map_elements` is much slower than the expression in Approach 2, which Polars evaluates on the whole column at once. Save it for the cases where nothing in the expression API will do the job.

We can drop `dr_ea_count` once we're done using it to maintain a neat table.

In [80]:
# Drop the "dr_ea_count" column
babynames = babynames.drop("dr_ea_count")
babynames.head(5)

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1986,"""Deandrea""",6
"""CA""","""F""",1988,"""Deandrea""",5
"""CA""","""F""",1990,"""Deandrea""",5
"""CA""","""F""",1994,"""Leandrea""",5
"""CA""","""M""",1985,"""Deandrea""",6


## Parting Note

The Polars library is enormous and contains many useful functions. Here is a link to its [documentation](https://docs.pola.rs/api/python/stable/reference/index.html). We certainly don't expect you to memorize each and every method of the library, and we will give you a reference sheet for exams.

Manipulating `DataFrame`s is not a skill that is mastered in just one day. The three custom sorts above all answer the same question, and none of them is the "real" one; trying several routes from point A to point B is how the syntax stops feeling arbitrary.

A goal of this course is to help you build your familiarity with the real-world programming practice of ... Googling! Answers to your questions can be found in documentation, Stack Overflow, and elsewhere. Being able to search for, read, and implement documentation is an important life skill for any data scientist.

Next, we will start digging deeper into the mechanics behind grouping data.